# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² dataset on second primary colorectal cancer using the `mlcroissant` library. All dataset elements are referenced by their `@id` as per the Croissant standard.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and inspect essential information using the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset object
dataset = mlc.Dataset(croissant_url)

# Access and display metadata
md = dataset.metadata
print(f"{md.name}: {md.description}\n")
print(f"Identifier: {md.identifier}")
print(f"Date Published: {md.datePublished}")
print(f"License: {md.license}")

## 2. Data Overview
List all available record sets and their fields using their `@id`. Croissant datasets may have multiple record sets (tables).

In [ ]:
# List all record sets by @id and their field @ids
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"\nRecord set '@id': {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            print(f"    - {fld['@id']} (name: {fld.get('name','')}, dataType: {fld.get('dataType', '')})")
    else:
        print("  (This record set contains no fields.)")

## 3. Data Extraction
Extract all records from the primary record set(s) as DataFrame(s).

_We will use the record set and field `@id`s found above to extract the data._

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("All record set @ids:")
for rsid in record_set_ids:
    print(" -", rsid)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nMain record set '@id': {main_record_set_id}")
    print("Columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering records with specific clinical characteristics, normalizing numeric variables (e.g., age), and grouping by attributes such as anatomical tumor site or MSI status.

_All columns are referenced by their Croissant field `@id`._

In [ ]:
# For this example, suppose 'age' is a numeric field with @id 'age_at_second_crc', and site @id is 'anatomical_location'
# If fields differ, please refer to the field list from the overview above.

df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs['@id'] == main_record_set_id:
        for fld in rs.get('field', []):
            # Try to auto-detect an age or relevant numeric field
            name = fld.get('name', '').lower() if 'name' in fld else ''
            if numeric_field_id is None and ('age' in name or fld.get('dataType','').lower() in ('integer','float') ):
                numeric_field_id = fld['@id']
            # Try to auto-detect anatomical group
            if group_field_id is None and ('anatomical' in name or 'site' in name):
                group_field_id = fld['@id']
        break

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Convert the numeric field to numeric dtype
if numeric_field_id and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter to records with numeric_field_id > 50 (age > 50 for example)
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id and compute mean and count if applicable
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std']).reset_index()
        print(f"Grouped '{numeric_field_id}' by '{group_field_id}': (showing up to 5 groups)")
        display(grouped_df.head())
    else:
        print("Group field not found; skipping grouping.")
else:
    print("Numeric field not found; cannot proceed with analysis.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and show a boxplot grouped by anatomical location if available.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        fig, ax = plt.subplots(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], ax=ax)
        ax.set_title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped - numeric field not available.")

## 6. Conclusion

- We loaded the FAIR² clinicopathological dataset using `mlcroissant` directly from its Croissant schema.
- We explored its record sets and fields using consistent `@id` references.
- Numeric and categoric clinical fields (such as age and anatomical site) were examined, normalized, and visualized.
- This pipeline can be extended for more specialized analyses or ML workflows, always referencing dataset fields by `@id` to ensure reproducibility and schema compliance.